In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
g=9.81
Ts=0.005


In [ ]:
class X():
    def __init__(self,state=np.zeros(6)):
        self.x,self.y,self.phi=state[0],state[1],state[2]
        self.xDot,self.yDot,self.phiDot=state[3],state[4],state[5]
    def __call__(self):
        return [self.x,self.y,self.phi,self.xDot,self.yDot,self.phiDot]


class Obstacle:
    def __init__(self,x,y,r):
        self.c=np.array([x,y])
        self.r=r
    def CheckCollisionThroughTime(self,state):
        nChecks=state.shape[1]
        for i in range(nChecks):
            currentState=X(state[:,i])
            if self.CheckCollision([currentState.x,currentState.y]):
                return 1
        return 0
    def CheckCollision(self,point,offset=0):
        distance=np.linalg.norm(np.array(point-self.c))
        if distance<=(self.r+offset):
            return 1
    def __call__(self):
        return self.c,self.r
    



In [ ]:
class DynamicObstacle:
    def __init__(self, x, y, r, vx=0.0, vy=0.0):
        self.c = np.array([float(x), float(y)])
        self.r = r
        self.v = np.array([float(vx), float(vy)])
        
    def update(self, dt, x_bounds=(0, 800), y_bounds=(0, 450)):
        self.c += self.v * dt
        
        if self.c[0] - self.r <= x_bounds[0] or self.c[0] + self.r >= x_bounds[1]:
            self.v[0] *= -1
        if self.c[1] - self.r <= y_bounds[0] or self.c[1] + self.r >= y_bounds[1]:
            self.v[1] *= -1

    def __call__(self):
        return self.c, self.r
    
    


In [ ]:
from scipy.integrate import solve_ivp
goalWidth=20
XLim=800
floor=0
cieling=450

class Quadcopter:
    def __init__(self, m=0.18, L=0.086, J=1.5e-4):
        self.m, self.L, self.J = m, L, J
        self.g = 9.81
        self.cd = 0.05 
        self.state = X()

    def step(self, action, dt, wind_force=np.array([0.0, 0.0])):
        u = np.clip(action, -1, 1)
        F = ((u[0] + 1)/2) * (1.8 * self.m * self.g)
        M = u[1] * 0.02

        s = np.array([self.state.x, self.state.y, self.state.phi, 
                      self.state.xDot, self.state.yDot, self.state.phiDot])
        
        def f(st, f_val, m_val):
            return np.array([
                st[3], st[4], st[5],
                -(f_val/self.m)*np.sin(st[2]) - (self.cd/self.m)*st[3] + (wind_force[0]/self.m),
                (f_val/self.m)*np.cos(st[2]) - self.g - (self.cd/self.m)*st[4] + (wind_force[1]/self.m),
                m_val/self.J
            ])

        k1 = f(s, F, M)
        k2 = f(s + 0.5*dt*k1, F, M)
        k3 = f(s + 0.5*dt*k2, F, M)
        k4 = f(s + dt*k3, F, M)
        
        self.state = X(s + (dt/6)*(k1 + 2*k2 + 2*k3 + k4))


In [ ]:
import control
A = np.zeros([6,6])
A[0,3]=1
A[1,4]=1
A[2,5]=1
A[3,2]=-9.81

m=0.18
J=1.5*10**-4
L=0.086

B=np.zeros([6,2])
B[4,0]=1/m
B[5,1]=1/J
print(A,B)
Q=np.diag([1,10,50,1,1,10])
R=np.diag([0.1,1])

sys_d = control.c2d(control.ss(A, B, np.eye(6), np.zeros((6,2))), Ts=Ts)
Ad, Bd = sys_d.A, sys_d.B
K, _, _ = control.dlqr(Ad, Bd, Q, R)

class Node():
    def __init__(self,parent=None,pos=None,g=0,h=0):
        self.parent = parent
        self.pos=pos
        self.g,self.h=g,h
        self.f=self.g+self.h
    def __eq__(self,other):
        return self.pos==other.pos
    def __hash__(self):
        return hash(self.pos)
    def __lt__(self,other):
        return self.f<other.f
    
class LQRController:
    def __init__(self,uEq,m,K=K):
        self.K=K
        self.uEq=uEq
        self.mass=m
    def Controller(self,desiredState,state,saturate=True):
        u=np.array(self.uEq)-self.K@(np.array(state())-np.array(desiredState()))
        #u[0]=u[0]/np.cos(state.phi)
        if saturate:
            u[0] = np.clip(u[0],0,1.8*self.mass*g)
            u[1] = np.clip(u[1],-1.0,1.0)
        return u
    
import heapq
class AStar:
    def __init__(self,res=1):
        self.gridWidth=int(XLim/res)
        self.gridHeight=int((cieling-floor)/res)
        self.res=res
    def CreateGrid(self,obstacles):
        grid=np.zeros((self.gridHeight,self.gridWidth))
        for r in range(self.gridHeight):
            for c in range(self.gridWidth):
                x,y=self.ConvertToWorld([r,c])
                point=np.array([x,y])

                for obstacle in obstacles:
                    if obstacle.CheckCollision(point,offset=5):
                        grid[r,c]=1
                        break
        print("Grid Created")
        self.grid=grid
    
    def ConvertToGrid(self,pos):
        x,y=pos
        c=int(round(x/self.res))
        r=int((round(y-floor)/self.res))
        c=max(0,min(c,self.gridWidth-1))
        r=max(0,min(self.gridHeight-1,r))
        return(r,c)
    
    def ConvertToWorld(self,pos):
        r,c=pos
        x=self.res*c+self.res/2
        y=floor+self.res*r+self.res/2
        return (x,y)
    
    def ReconstructPath(self,endNode):
        path=[]
        node=endNode
        while node is not None:
            path.append(node.pos)
            node=node.parent
        path=path[::-1]
        pathWorld=[self.ConvertToWorld(pos) for pos in path]
        return pathWorld
    def Heuristic(self,start,goal):
        (r1, c1) = start
        (r2, c2) = goal
        (r3,c3)=self.startPos
        cross=abs((r1-r2)*(c1-c2)-(r3-r2)*(c3-c2))
        return np.sqrt((r1 - r2)**2 + (c1 - c2)**2)+0.0005*cross
    def Valid(self,pos):
        r,c=pos
        if 0 <= r < self.gridHeight and 0 <= c < self.gridWidth:
            return self.grid[r, c] == 0
        return 0
    
    def PathFind(self,start,end):
        end=self.ConvertToGrid(end)
        start=self.ConvertToGrid(start)
        self.startPos=start
        startNode=Node(None,start,h=self.Heuristic(start,end))
        
        goalNode=Node(None,end)
        open=list()
        heapq.heappush(open, (startNode.f, startNode))
        gs = {start: 0}

        
        while open:
            _,cNode=heapq.heappop(open)
            if cNode.g>gs.get(cNode.pos,float('inf')):
                continue
            if cNode == goalNode:
                return self.ReconstructPath(cNode)
            
            for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0:
                            continue 
                        neighbor_pos = (cNode.pos[0] + dr, cNode.pos[1] + dc)
                        if self.Valid(neighbor_pos):
                            move_cost = np.sqrt(2) if dr != 0 and dc != 0 else 1
                            new_g = cNode.g + move_cost

                            if new_g < gs.get(neighbor_pos, float('inf')):
                                gs[neighbor_pos] = new_g
                                new_h = self.Heuristic(neighbor_pos, end)
                                neighbor_node = Node(cNode, neighbor_pos, new_g, new_h)
                                heapq.heappush(open, (neighbor_node.f, neighbor_node))


from scipy.interpolate import make_interp_spline
class MinimumSnapTrajectory:
    def __init__(self, path, avgSpeed=1):
        path=np.array(path)
        _, idx = np.unique(path, axis=0, return_index=True)
        path = path[np.sort(idx)]
        if len(path) > 2:
            filtered_path = [path[0]]
            for i in range(1, len(path)-1):
                dist = np.linalg.norm(path[i] - filtered_path[-1])
                if dist > 5.0: 
                    filtered_path.append(path[i])
            filtered_path.append(path[-1]) 
            self.path = np.array(filtered_path)
        else:
            self.path = path
        diffs = np.linalg.norm(np.diff(self.path, axis=0), axis=1)
        self.times = np.cumsum(diffs) / avgSpeed
        self.times = np.insert(self.times, 0, 0)
        self.totalTime = self.times[-1]
        self.splineX = make_interp_spline(self.times, self.path[:, 0], k=5)
        self.splineY = make_interp_spline(self.times, self.path[:, 1], k=5)
    def __call__(self,t):
        t = np.clip(t, 0, self.totalTime)
        x=self.splineX(t)
        y=self.splineY(t)
        xDot=self.splineX(t, nu=1)
        yDot=self.splineY(t, nu=1)
        return [x,y,0,xDot,yDot,0]
    def isDone(self,time):
        return time>=self.totalTime

class SimulationEnviroment:
    def __init__(self,goal=[250,200],res=1):
        self.obstacle=list()
        self.goal=Obstacle(goal[0],goal[1],goalWidth)
        self.pathFinder=AStar(res)
    def CreateObstacle(self,x,y,r):
        self.obstacle.append(Obstacle(x,y,r))
    def checkSolution(self,solution):
        if (solution[1]<floor).any():
            print("Crashed into the ground")
        elif (solution[1]>cieling).any():
            print("Crashed into the cieling")
        for obstacle in self.obstacle:
            if obstacle.CheckCollisionThroughTime(solution):
                print("Crashed into an obstacle")
                #return None
        if self.goal.CheckCollisionThroughTime(solution):
            print("Reached Goal")
        if (solution[0]>XLim).any() or (solution[0]<0).any() :
            print("Crashed into the walls")
    def PlotObjects(self,ax):
        for obstacle in self.obstacle:
            c,r=obstacle()
            circle=plt.Circle((c[0],c[1]),r,color="red",alpha=0.7)
            ax.add_patch(circle)
        c,r=self.goal()   
        circle=plt.Circle((c[0],c[1]),r,color="green",alpha=0.7)
        ax.add_patch(circle)
  
    def distancePointSeg(self,p,a,b):
        p,a,b=np.array(p),np.array(a),np.array(b)
        ab=b-a
        ap=p-a
        if np.linalg.norm(ab)==0: return np.linalg.norm(ap)
        t=np.clip(np.dot(ap,ab)/np.dot(ab,ab),0,1)
        closest=a+t*ab
        return np.linalg.norm(p-closest)
    
    def lineSafe(self,start,end):
        for obstacle in self.obstacle:
            c,r=obstacle()
            dist=self.distancePointSeg(c,start,end)
            if dist<(r+5):
                return False
        return True

        
    def CreatePath(self):
        self.pathFinder.CreateGrid(self.obstacle)
        referencePath=self.pathFinder.PathFind([0,0],self.goal()[0])
        referencePath=self.sampledPath(self.PullPath(referencePath))
        return referencePath
            
    def PullPath(self,path):
        pulledPath=[path[0]]
        idx=0
        while idx<len(path)-1:
            for lookIdx in range(len(path)-1,idx,-1):
                startNode=path[idx]
                endNode=path[lookIdx]
                if self.lineSafe(startNode,endNode):
                    pulledPath.append(endNode)
                    idx=lookIdx
                    break
        return pulledPath

    def sampledPath(self,path,segLength=20):
        dPath=[path[0]]
        for i in range(len(path) - 1):
            start = np.array(path[i])
            end = np.array(path[i+1])
            dist = np.linalg.norm(end - start)
            if dist > segLength:
                num_points = int(np.ceil(dist / segLength))
                for j in range(1, num_points + 1):
                    new_point = start + (end - start) * (j / num_points)
                    dPath.append(new_point)
            else:
                dPath.append(end)
                
        return dPath


In [ ]:
from cvxopt import matrix,solvers

solvers.options['show_progress']=False

class Sigma:
    def __init__(self, k=0.02, limit=100.0):
        self.k = k
        self.limit = limit
    def val(self, s):
        return self.limit * np.tanh(self.k * s)
    def prime(self, s):
        return self.limit * self.k * (1 - np.tanh(self.k * s)**2)
    def dPrime(self, s):
        tanh_s = np.tanh(self.k * s)
        return -2 * self.limit * (self.k**2) * tanh_s * (1 - tanh_s**2)

class CBFQPController:
    def __init__(self, nominal_controller, m, J, gamma=0.8, alpha=0.8):
        self.nominal = nominal_controller 
        self.m = m
        self.J = J
        self.gamma = gamma # Lookahead horizon
        self.alpha = alpha 
        self.sigma = Sigma() 

    def cbfConstraint(self, state, obstacles, gp_mu=np.zeros(3),beta=2.0):
        G_obs, H_obs = [], []
        for obs in obstacles:
            rSafe = obs.r + 8
            rx, ry = state.x - obs.c[0], state.y - obs.c[1]
           
            s = rx * np.sin(state.phi) - ry * np.cos(state.phi)
            p = rx * np.cos(state.phi) + ry * np.sin(state.phi)
            
            v = -state.xDot * np.sin(state.phi) + state.yDot * np.cos(state.phi)
            w =  state.xDot * np.cos(state.phi) + state.yDot * np.sin(state.phi)
            
            ds = -v + p * state.phiDot 
            
            h = (rx**2 + ry**2) - rSafe**2
            hDot = 2 * (rx * state.xDot + ry * state.yDot)
            sig = self.sigma.val(s)
            sigP = self.sigma.prime(s)
            sigPP = self.sigma.dPrime(s)
            
            gHat = h - sig
            gHatDot = hDot - sigP * ds
            hAug = gHatDot + self.gamma * gHat

            C_drift = 2 * (state.xDot**2 + state.yDot**2) - 2 * 9.81 * ry
            C_sig = -sigPP * (ds**2) - sigP * (9.81 * np.cos(state.phi) + 2 * w * state.phiDot + s * (state.phiDot**2))
            C = C_drift + C_sig + self.gamma * gHatDot
            
            
        
            
            Lg1 = -(1 / self.m) * (2 * s - sigP)
            Lg2 = -(p * sigP) / self.J
            
            G_obs.append([-Lg1, -Lg2])
            H_obs.append(C + self.alpha * hAug)
            
        G_bounds, H_bounds = [], []
        
        # Floor
        y_floor = 15
        h_floor = state.y - y_floor
        Lg_floor = [-np.cos(state.phi) / self.m, 0]
        gamma_floor=0.4
        boundary_floor = -9.81 + 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_floor
        G_bounds.append(Lg_floor)
        H_bounds.append(boundary_floor)
            
        # Ceiling
        y_ceil = 445
        h_ceil = y_ceil - state.y
        Lg_ceil = [np.cos(state.phi) / self.m, 0]
        boundary_ceil = 9.81 - 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_ceil
        G_bounds.append(Lg_ceil)
        H_bounds.append(boundary_ceil)

        return np.array(G_obs), np.array(H_obs), np.array(G_bounds), np.array(H_bounds)

    def filter_action(self, u_nom, state, obstacles):
        P = matrix(np.diag([1, 1, 1e8])) 
        q = matrix([-u_nom[0], -u_nom[1], 0])
            
        G_obs, h_obs, G_bounds, h_bounds = self.cbfConstraint(state, obstacles)
        
        max_thrust = 1.8 * self.m * 9.81
        max_moment = 0.02
        u_max = np.array([max_thrust, max_moment])
        u_min = np.array([0, -max_moment])
        
        G_sat = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]])
        h_sat = np.array([u_max[0], -u_min[0], u_max[1], -u_min[1]])

        if len(G_obs) > 0:
            slack_col = -np.ones((len(G_obs), 1))
            G_obs_cbf = np.hstack([G_obs, slack_col])
            
            no_slack_bounds = np.zeros((len(G_bounds), 1))
            G_bounds_cbf = np.hstack([G_bounds, no_slack_bounds])
            
            no_slack_sat = np.zeros((4, 1))
            G_sat_cbf = np.hstack([G_sat, no_slack_sat])
            
            G_final = matrix(np.vstack([G_obs_cbf, G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_obs, h_bounds, h_sat]).astype(np.float64))
        else:
            no_slack_bounds = np.zeros((len(G_bounds), 1))
            G_bounds_cbf = np.hstack([G_bounds, no_slack_bounds])
            no_slack_sat = np.zeros((4, 1))
            G_sat_cbf = np.hstack([G_sat, no_slack_sat])
            
            G_final = matrix(np.vstack([G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_bounds, h_sat]).astype(np.float64))

        try:
            sol = solvers.qp(P, q, G_final, h_final)
            if sol['status'] != 'optimal':
                raise ValueError("QP not optimal")
            
            u_safe = np.array(sol['x']).flatten()[:2]
            
            slack_vals = np.array(sol['x']).flatten()[2:]
            if len(slack_vals) > 0 and np.max(slack_vals) > 5.0:
                raise ValueError("Constraint conflict - Slack threshold exceeded")
                
            return u_safe
            
        except Exception:
            target_phi = np.clip(state.xDot * 0.05, -np.pi/3, np.pi/3)
            M_fallback = 0.5 * (target_phi - state.phi) - 0.1 * state.phiDot
            M_fallback = np.clip(M_fallback, -max_moment, max_moment)
            
            F_fallback = max_thrust
            
            return np.array([F_fallback, M_fallback])


In [ ]:
from cvxopt import matrix, solvers
solvers.options['show_progress'] = False

class DynCBFQPController:
    def __init__(self, nominal_controller, m, J, gamma=1.3, alpha=1.2):
        self.nominal = nominal_controller 
        self.m = m
        self.J = J
        self.gamma = gamma 
        self.alpha = alpha 
        self.sigma = Sigma() 

    def cbfConstraint(self, state, obstacles):
        G_obs, H_obs = [], []
        
        for obs in obstacles:
            rSafe = obs.r + 8 
            rx = state.x - obs.c[0]
            ry = state.y - obs.c[1]
            
            obs_vx = getattr(obs, 'v', np.array([0.0, 0.0]))[0]
            obs_vy = getattr(obs, 'v', np.array([0.0, 0.0]))[1]
            
            rel_xDot = state.xDot - obs_vx
            rel_yDot = state.yDot - obs_vy
            
            s = rx * np.sin(state.phi) - ry * np.cos(state.phi)
            p = rx * np.cos(state.phi) + ry * np.sin(state.phi)
            
            v_rel = -rel_xDot * np.sin(state.phi) + rel_yDot * np.cos(state.phi)
            w_rel =  rel_xDot * np.cos(state.phi) + rel_yDot * np.sin(state.phi)
            
            ds = -v_rel + p * state.phiDot 
            
            h = (rx**2 + ry**2) - rSafe**2
            hDot = 2 * (rx * rel_xDot + ry * rel_yDot)
            
            sig = self.sigma.val(s)
            sigP = self.sigma.prime(s)
            sigPP = self.sigma.dPrime(s)
            
            gHat = h - sig
            gHatDot = hDot - sigP * ds
            hAug = gHatDot + self.gamma * gHat
            
            C_drift = 2 * (rel_xDot**2 + rel_yDot**2) - 2 * 9.81 * ry
            C_sig = -sigPP * (ds**2) - sigP * (9.81 * np.cos(state.phi) + 2 * w_rel * state.phiDot + s * (state.phiDot**2))
            
            C = C_drift + C_sig + self.gamma * gHatDot
            
            Lg1 = -(1 / self.m) * (2 * s + sigP) 
            Lg2 = -(p * sigP) / self.J
            
            G_obs.append([-Lg1, -Lg2])
            H_obs.append(C + self.alpha * hAug)
            
        G_bounds, H_bounds = [], []
        
        y_floor = 15
        h_floor = state.y - y_floor
        Lg_floor = [-np.cos(state.phi) / self.m, 0]
        gamma_floor = 0.4
        boundary_floor = -9.81 + 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_floor
        G_bounds.append(Lg_floor)
        H_bounds.append(boundary_floor)
            
        y_ceil = 445
        h_ceil = y_ceil - state.y
        Lg_ceil = [np.cos(state.phi) / self.m, 0]
        boundary_ceil = 9.81 - 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_ceil
        G_bounds.append(Lg_ceil)
        H_bounds.append(boundary_ceil)

        return np.array(G_obs), np.array(H_obs), np.array(G_bounds), np.array(H_bounds)

    def filter_action(self, u_nom, state, obstacles):
        P = matrix(np.diag([1, 1, 1e8])) 
        q = matrix([-u_nom[0], -u_nom[1], 0])
        
        G_obs, h_obs, G_bounds, h_bounds = self.cbfConstraint(state, obstacles)
        
        max_thrust = 1.8 * self.m * 9.81
        max_moment = 0.02
        u_max = np.array([max_thrust, max_moment])
        u_min = np.array([0, -max_moment])
        
        G_sat = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]])
        h_sat = np.array([u_max[0], -u_min[0], u_max[1], -u_min[1]])

        if len(G_obs) > 0:
            slack_col = -np.ones((len(G_obs), 1))
            G_obs_cbf = np.hstack([G_obs, slack_col])
            no_slack_bounds = np.zeros((len(G_bounds), 1))
            G_bounds_cbf = np.hstack([G_bounds, no_slack_bounds])
            no_slack_sat = np.zeros((4, 1))
            G_sat_cbf = np.hstack([G_sat, no_slack_sat])
            
            G_final = matrix(np.vstack([G_obs_cbf, G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_obs, h_bounds, h_sat]).astype(np.float64))
        else:
            no_slack_bounds = np.zeros((len(G_bounds), 1))
            G_bounds_cbf = np.hstack([G_bounds, no_slack_bounds])
            no_slack_sat = np.zeros((4, 1))
            G_sat_cbf = np.hstack([G_sat, no_slack_sat])
            
            G_final = matrix(np.vstack([G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_bounds, h_sat]).astype(np.float64))

        try:
            sol = solvers.qp(P, q, G_final, h_final)
            if sol['status'] != 'optimal':
                raise ValueError("QP not optimal")
            
            u_safe = np.array(sol['x']).flatten()[:2]
            slack_vals = np.array(sol['x']).flatten()[2:]
            
            if len(slack_vals) > 0 and np.max(slack_vals) > 2.5:
                raise ValueError("Constraint conflict")
            return u_safe
            
        except Exception:
            target_phi = np.clip(state.xDot * 0.1, -np.pi/3, np.pi/3)
            M_fallback = 1.2 * (target_phi - state.phi) - 0.3 * state.phiDot
            M_fallback = np.clip(M_fallback, -max_moment, max_moment)
            F_fallback = max_thrust
            return np.array([F_fallback, M_fallback])


In [ ]:
from cvxopt import matrix, solvers
import numpy as np

solvers.options['show_progress'] = False

class AdaptiveCBFQPController:
    def __init__(self, m_nom=0.18, J=1.5e-4, gamma=1.0, alpha=1.0):
        self.J = J
        self.gamma = gamma 
        self.alpha = alpha 
        self.sigma = Sigma() 
        
        self.theta_m = 1/ m_nom  
        self.d_x = 0              
        self.d_y = 0
        self.theta_cd = 0          
        
        self.Gamma_m = 0.5
        self.Gamma_w = 0.16  
        self.Gamma_cd = 0.02 
        
        self.theta_min = 1/ 0.35  
        self.theta_max = 1/ 0.10  
        self.wind_max = 15         
        self.cd_max = 2   

    def cbfConstraint(self, state, obstacles):
        G_obs, H_obs = [], []
        for obs in obstacles:
            rSafe = obs.r + 8
            rx, ry = state.x - obs.c[0], state.y - obs.c[1]
            
            obs_vx = getattr(obs, 'v', np.array([0.0, 0.0]))[0]
            obs_vy = getattr(obs, 'v', np.array([0.0, 0.0]))[1]
            rel_xDot = state.xDot - obs_vx
            rel_yDot = state.yDot - obs_vy

            s = rx * np.sin(state.phi) - ry * np.cos(state.phi)
            p = rx * np.cos(state.phi) + ry * np.sin(state.phi)
            v_rel = -rel_xDot * np.sin(state.phi) + rel_yDot * np.cos(state.phi)
            w_rel =  rel_xDot * np.cos(state.phi) + rel_yDot * np.sin(state.phi)
            
            ds = -v_rel + p * state.phiDot
            h = (rx**2 + ry**2) - rSafe**2
            hDot = 2 * (rx * rel_xDot + ry * rel_yDot)
            
            sig = self.sigma.val(s)
            sigP = self.sigma.prime(s)
            sigPP = self.sigma.dPrime(s)
            
            gHat = h - sig
            gHatDot = hDot - sigP * ds
            hAug = gHatDot + self.gamma * gHat

            C_drift = 2 * (rel_xDot**2 + rel_yDot**2) - 2 * 9.81 * ry
            C_sig = -sigPP * (ds**2) - sigP * (9.81 * np.cos(state.phi) + 2 * w_rel * state.phiDot + s * (state.phiDot**2))
            
            total_dist_x = self.d_x - self.theta_cd * state.xDot
            total_dist_y = self.d_y - self.theta_cd * state.yDot
            
            C_dist = total_dist_x * (2 * rx + sigP * np.sin(state.phi)) + \
                     total_dist_y * (2 * ry - sigP * np.cos(state.phi)) 
            
            C = C_drift + C_sig + C_dist + self.gamma * gHatDot
            
            Lg1 = -self.theta_m * (2 * s - sigP)
            Lg2 = -(p * sigP) / self.J
            
            G_obs.append([-Lg1, -Lg2])
            H_obs.append(C + self.alpha * hAug)
            
        G_bounds, H_bounds = [], []
        y_floor, h_floor = 15, state.y - 15
        Lg_floor = [-self.theta_m * np.cos(state.phi), 0]
        gamma_floor = 0.4
        G_bounds.append(Lg_floor)
        H_bounds.append(-9.81 + 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_floor)
            
        y_ceil, h_ceil = 445, 445 - state.y
        Lg_ceil = [self.theta_m * np.cos(state.phi), 0]
        G_bounds.append(Lg_ceil)
        H_bounds.append(9.81 - 2 * gamma_floor * state.yDot + (gamma_floor**2) * h_ceil)

        return np.array(G_obs), np.array(H_obs), np.array(G_bounds), np.array(H_bounds)

    def filter_action(self, u_nom, state, obstacles):
        P = matrix(np.diag([1.0, 1.0, 1e8])) 
        q = matrix([-float(u_nom[0]), -float(u_nom[1]), 0.0])
        
        G_obs, h_obs, G_bounds, h_bounds = self.cbfConstraint(state, obstacles)
        
        max_thrust = 1.8 * 0.18 * 9.81
        max_moment = 0.02
        u_max = np.array([max_thrust, max_moment])
        u_min = np.array([0, -max_moment])
        
        G_sat = np.array([[1, 0], [-1, 0], [0, 1], [0, -1]])
        h_sat = np.array([u_max[0], -u_min[0], u_max[1], -u_min[1]])

        if len(G_obs) > 0:
            slack_col = -np.ones((len(G_obs), 1))
            G_obs_cbf = np.hstack([G_obs, slack_col])
            G_bounds_cbf = np.hstack([G_bounds, np.zeros((len(G_bounds), 1))])
            G_sat_cbf = np.hstack([G_sat, np.zeros((4, 1))])
            
            G_final = matrix(np.vstack([G_obs_cbf, G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_obs, h_bounds, h_sat]).astype(np.float64))
        else:
            G_bounds_cbf = np.hstack([G_bounds, np.zeros((len(G_bounds), 1))])
            G_sat_cbf = np.hstack([G_sat, np.zeros((4, 1))])
            
            G_final = matrix(np.vstack([G_bounds_cbf, G_sat_cbf]).astype(np.float64))
            h_final = matrix(np.hstack([h_bounds, h_sat]).astype(np.float64))

        try:
            sol = solvers.qp(P, q, G_final, h_final)
            if sol['status'] != 'optimal': raise ValueError("QP not optimal")
            return np.array(sol['x']).flatten()[:2]
        except Exception:
            target_phi = np.clip(state.xDot * 0.1, -np.pi/3, np.pi/3)
            M_fallback = 1* (target_phi - state.phi) - 0.2 * state.phiDot
            return np.array([max_thrust, np.clip(M_fallback, -max_moment, max_moment)])

    def adapt(self, phi, u1_applied, true_ax, true_ay, dt, vx, vy):        
        expected_ax = -self.theta_m * u1_applied * np.sin(phi) + self.d_x - self.theta_cd * vx
        expected_ay =  self.theta_m * u1_applied * np.cos(phi) - 9.81 + self.d_y - self.theta_cd * vy
        
        error_ax = true_ax - expected_ax
        error_ay = true_ay - expected_ay
        
        grad_theta_m = error_ay * (u1_applied * np.cos(phi)) 
        
        grad_dx = error_ax
        grad_dy = error_ay
        
        grad_theta_cd = error_ax * (-vx) + error_ay * (-vy)
        
        self.theta_m += (self.Gamma_m) * grad_theta_m * dt
        self.theta_m = np.clip(self.theta_m, self.theta_min, self.theta_max)
        
        self.d_x += (self.Gamma_w) * grad_dx * dt
        self.d_x = np.clip(self.d_x, -self.wind_max, self.wind_max)
        
        self.d_y += (self.Gamma_w) * grad_dy * dt
        self.d_y = np.clip(self.d_y, -self.wind_max, self.wind_max)
        
        self.theta_cd += (self.Gamma_cd) * grad_theta_cd * dt
        self.theta_cd = np.clip(self.theta_cd, 0.0, self.cd_max) 


In [ ]:
class TimeOptimalNavEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.quad = Quadcopter()
        self.dt = 0.02
        self.max_steps = 4000

        self.bounds = np.array([0, 800, 0, 450])
        self.goal_radius = 25

        self.alpha = 10       # shaping strength
        self.R_goal = 2000
        self.R_crash = 1000

        self.n_lidar = 8

        self.action_space = spaces.Box(-1, 1, (2,), np.float32)
        self.observation_space = spaces.Box(-1, 1, (7 + self.n_lidar,), np.float32)
        
    def _get_obs(self):
        s = self.quad.state

        rel = (self.goal - np.array([s.x, s.y])) / 800
        vel = np.array([s.xDot, s.yDot]) / 10

        ang = np.array([np.sin(s.phi), np.cos(s.phi)])
        ang_vel = np.array([s.phiDot / 5])

        lidar = self._get_lidar()

        return np.concatenate([rel, vel, ang, ang_vel, lidar]).astype(np.float32)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.steps = 0

        # random start and goal
        self.start = np.random.uniform([10, 10], [750, 425])
        self.goal = np.random.uniform([10, 10], [750, 425])

        self.quad.state = X(np.array([self.start[0], self.start[1], 0, 0, 0, 0]))
        self.prev_dist = np.linalg.norm(self.goal - self.start)

        return self._get_obs(), {}
    def _get_lidar(self):
        return np.ones(self.n_lidar, dtype=np.float32)
    def step(self, action, wind_force=np.array([0.0, 0.0])):
        self.steps += 1        
        self.quad.step(action, self.dt, wind_force=wind_force)

        s = self.quad.state
        pos = np.array([s.x, s.y])
        dist = np.linalg.norm(self.goal - pos)

        # reward section
        reward = -1
        reward += self.alpha * (self.prev_dist - dist)
        self.prev_dist = dist

        terminated = False
        info = {'is_success': False} 

        # Goal reached
        if dist < self.goal_radius:
            reward += self.R_goal
            terminated = True
            info['is_success'] = True 

        # Crash check
        if not (-1 <= s.x <= 1000 and -1 <= s.y <= 450):
            reward -= self.R_crash
            terminated = True

        truncated = self.steps >= self.max_steps

        truncated = self.steps >= self.max_steps
        return self._get_obs(), reward, terminated, truncated, info


In [ ]:
from stable_baselines3 import SAC
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import VecMonitor
def make_env():
    return Monitor(TimeOptimalNavEnv())

env = make_vec_env(make_env, n_envs=16, vec_env_cls=SubprocVecEnv)

env = VecMonitor(env)

policy_kwargs = dict(
    net_arch=[256, 256]
)

model = SAC(
    "MlpPolicy",
    env,
    learning_rate=3e-4,
    buffer_size=2_000_000,
    learning_starts=25_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    train_freq=64,
    gradient_steps=64,
    ent_coef="auto",         
    policy_kwargs=policy_kwargs,
    verbose=1,
)
model = model.load("Base-Straight-Line-Model")


In [ ]:
import casadi as ca
import numpy as np

class QuadcopterNMPC:
    def __init__(self, m=0.18, J=1.5e-4, dt=0.02, N=25, num_obs=5):
        self.m = m
        self.J = J
        self.g = 9.81
        self.dt = dt
        self.N = N  
        self.num_obs = num_obs 
        
        self.max_thrust = 1.8 * self.m * self.g
        self.max_moment = 0.02
        
        self._setup_optimizer()

    def _setup_optimizer(self):
        self.opti = ca.Opti()

        self.X = self.opti.variable(6, self.N + 1) 
        self.U = self.opti.variable(2, self.N)     
        
        self.X0 = self.opti.parameter(6)    
        self.X_ref = self.opti.parameter(6) 
        self.obstacles = self.opti.parameter(5, self.num_obs) 

        self.opti.subject_to(self.X[:, 0] == self.X0)
        
        cost = 0
        u_eq = ca.vertcat(self.m * self.g, 0.0) 
        
        Q = np.diag([10.0, 10.0, 1.0, 1.0, 1.0, 0.1]) 
        R = np.diag([0.1, 5.0])
        
        for k in range(self.N):
            self.opti.subject_to(self.U[0, k] >= 0.0)
            self.opti.subject_to(self.U[0, k] <= self.max_thrust)
            self.opti.subject_to(self.U[1, k] >= -self.max_moment)
            self.opti.subject_to(self.U[1, k] <= self.max_moment)
            
            x_k = self.X[:, k]
            u_k = self.U[:, k]
            x_next = x_k + self.dt * self._dynamics(x_k, u_k)
            self.opti.subject_to(self.X[:, k+1] == x_next)
            
            state_err = self.X[:, k] - self.X_ref
            ctrl_err = self.U[:, k] - u_eq
            cost += ca.mtimes([state_err.T, Q, state_err]) + ca.mtimes([ctrl_err.T, R, ctrl_err])
            
            cost += 5000* ca.fmax(0, ca.fabs(self.X[2, k+1]) - 0.5)**2
            cost += 5000* ca.fmax(0, 15- self.X[1, k+1])**2
            
            for j in range(self.num_obs):
                obs_x_future = self.obstacles[0, j] + ((k + 1) * self.dt) * self.obstacles[3, j]
                obs_y_future = self.obstacles[1, j] + ((k + 1) * self.dt) * self.obstacles[4, j]
                
                
                obs_r = self.obstacles[2, j] 
                sensing_radius = obs_r + 25
                
                dist = ca.sqrt((self.X[0, k+1] - obs_x_future)**2 + (self.X[1, k+1] - obs_y_future)**2 + 1e-3) 
                
                cost += 1500* ca.fmax(0, sensing_radius - dist)**2       

        term_err = self.X[:, self.N] - self.X_ref
        cost += ca.mtimes([term_err.T, Q * 5, term_err])
        
        self.opti.minimize(cost)

        p_opts = {"expand": True, "print_time": False}
        s_opts = {"max_iter": 50, "print_level": 0, "acceptable_tol": 1e-2}
        self.opti.solver("ipopt", p_opts, s_opts)

    def _dynamics(self, x, u):
        return ca.vertcat(
            x[3],
            x[4],
            x[5],
            -(u[0] / self.m) * ca.sin(x[2]),
            (u[0] / self.m) * ca.cos(x[2]) - self.g,
            u[1] / self.J
        )

    def compute_action(self, current_state, target_pos, active_obstacles):
        state_vec = np.array([current_state.x, current_state.y, current_state.phi, 
                              current_state.xDot, current_state.yDot, current_state.phiDot])
        self.opti.set_value(self.X0, state_vec)
        
        dx = target_pos[0] - current_state.x
        dy = target_pos[1] - current_state.y
        dist = np.sqrt(dx**2 + dy**2)
        
        
        if dist > 80.0:
            local_target_x = current_state.x + (dx / dist) * 80.0
            local_target_y = current_state.y + (dy / dist) * 80.0
        else:
            local_target_x = target_pos[0]
            local_target_y = target_pos[1]
            
        ref_vec = np.array([local_target_x, local_target_y, 0.0, 0.0, 0.0, 0.0])
        self.opti.set_value(self.X_ref, ref_vec)
        
        obs_matrix = np.zeros((5, self.num_obs))
        for i, obs in enumerate(active_obstacles[:self.num_obs]):
            obs_matrix[0, i] = obs.c[0]
            obs_matrix[1, i] = obs.c[1]
            obs_matrix[2, i] = obs.r
            obs_matrix[3, i] = obs.v[0] 
            obs_matrix[4, i] = obs.v[1] 
        self.opti.set_value(self.obstacles, obs_matrix)
        
        if not hasattr(self, 'initialized'):
            for k in range(self.N + 1):
                self.opti.set_initial(self.X[:, k], state_vec)
            self.opti.set_initial(self.U[0, :], self.m * self.g)
            self.opti.set_initial(self.U[1, :], 0.0)
            self.initialized = True
            
        try:
            sol = self.opti.solve()
            opt_u = sol.value(self.U[:, 0]) 
            
            self.opti.set_initial(self.X, sol.value(self.X))
            self.opti.set_initial(self.U, sol.value(self.U))
            
            return opt_u
            
        except Exception:
            opt_u = self.opti.debug.value(self.U[:, 0])
            if np.any(np.isnan(opt_u)):
                return np.array([self.m * self.g, 0.0])
            return opt_u


In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib
import time
import numpy as np
import copy
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.animation as animation
from IPython.display import HTML
import matplotlib


def get_scenario(scenario_name):
    
    start = [0, 0]
    goal = [750, 200]
    mass_func = lambda t: 0.18
    wind_func = lambda t: np.array([0.0, 0.0])
    obstacles = []
    
    if scenario_name == "Static_Single":
        obstacles = [DynamicObstacle(400, 100, 80, vx=0.0, vy=0.0)]
        
    elif scenario_name == "Static_Multi":
        obstacles = [
            DynamicObstacle(350, 50, 70, vx=0.0, vy=0.0), 
            DynamicObstacle(650, 225, 30, vx=0.0, vy=0.0),
            DynamicObstacle(350, 175, 50, vx=0.0, vy=0.0), 
            DynamicObstacle(500, 350, 50, vx=0.0, vy=0.0),
            DynamicObstacle(200, 250, 75, vx=0.0, vy=0.0), 
            DynamicObstacle(600, 50, 115, vx=0.0, vy=0.0)
        ]
        
    elif scenario_name == "Dynamic_Single":
        start = [50, 225]
        obstacles = [DynamicObstacle(400, 225, 80, vx=0.0, vy=15.0)]
        
    elif scenario_name == "Dynamic_Weave":
        start = [50, 225]
        obstacles = [
            DynamicObstacle(300, 100, 50, vx=0.0, vy=15.0),
            DynamicObstacle(500, 350, 50, vx=0.0, vy=-15.0),
            DynamicObstacle(400, 225, 40, vx=10.6, vy=10.6)
        ]

    elif scenario_name == "Dynamic_Hard":
        start = [50, 225]
        obstacles = [
            DynamicObstacle(350, 70,  60, vx=0.0, vy=15.0),    
            DynamicObstacle(650, 225, 40, vx=0.0, vy=-15.0),
            DynamicObstacle(400, 175, 50, vx=-10.6, vy=10.6), 
            DynamicObstacle(200, 350, 40, vx=15.0, vy=0.0),  
            DynamicObstacle(200, 50,  40, vx=6.7, vy=13.4)     
        ]
        
    elif scenario_name == "Loss_of_Parameter_1":
        obstacles = [DynamicObstacle(400, 100, 80, vx=0.0, vy=0.0)]
        mass_func = lambda t: 0.28
        wind_func = lambda t: np.array([0.15, 0.0])
        
    elif scenario_name == "Dynamic_Shift":
        start = [50, 225]
        obstacles = [DynamicObstacle(400, 225, 80, vx=0.0, vy=0.0)]
        mass_func = lambda t: 0.1 if t < 8else 0.3
        
        def wind_shift(t):
            w_start = np.array([0.0, -0.25])
            w_end = np.array([-0.35, 0.15])
            # Sigmoid centered at t=8s, spread over roughly 2 seconds
            alpha = 1/ (1+ np.exp(-4* (t - 8.0))) 
            return w_start * (1 - alpha) + w_end * alpha
            
        wind_func = wind_shift
    
    elif scenario_name == "Dynamic_Hard_Uncertainty":
        start = [50, 225]
        obstacles = [
            DynamicObstacle(350, 70,  60, vx=0.0, vy=15.0),    
            DynamicObstacle(650, 225, 40, vx=0.0, vy=-15.0),
            DynamicObstacle(400, 175, 50, vx=-10.6, vy=10.6), 
            DynamicObstacle(200, 350, 40, vx=15.0, vy=0.0),  
            DynamicObstacle(200, 50,  40, vx=6.7, vy=13.4)     
        ]
        
        mass_func = lambda t: 0.31
        
        def wind_shift(t):
            w_start = np.array([0.0, -0.1])
            w_end = np.array([0.2, 0.2]) 
            # Sigmoid centered at t=16s, spread over roughly 2 seconds
            alpha = 1/ (1+ np.exp(-4* (t - 16))) 
            return w_start * (1 - alpha) + w_end * alpha
            
        wind_func = wind_shift

    return {'start': start, 'goal': goal, 'obstacles': obstacles, 'mass_func': mass_func, 'wind_func': wind_func}

import control

def run_simulation(controller_type, scenario_name, model_rl=None):
    scenario_dict = get_scenario(scenario_name)
    env = TimeOptimalNavEnv()
    start, goal = scenario_dict['start'], scenario_dict['goal']
    active_obstacles = [copy.deepcopy(o) for o in scenario_dict['obstacles']]
    
    cbf, nmpc, lqr, traj = None, None, None, None
    
    if controller_type == "RL-CBF": 
        cbf = CBFQPController(nominal_controller=None, m=0.18, J=env.quad.J, gamma=0.8, alpha=0.8)
    elif controller_type == "RL-aCBF": 
        cbf = AdaptiveCBFQPController(m_nom=0.18, J=env.quad.J, gamma=0.6, alpha=0.6)
    elif controller_type == "RL-DynCBF":
        cbf = DynCBFQPController(nominal_controller=None, m=0.18, J=env.quad.J, gamma=0.8, alpha=0.8)
    elif controller_type in ["NMPC","NMPC_A*"]: 
        nmpc = QuadcopterNMPC(m=0.18, J=env.quad.J, dt=env.dt, N=62, num_obs=len(active_obstacles))
    elif controller_type == "NMPC-CBF": 
        cbf = CBFQPController(nominal_controller=None, m=0.18, J=env.quad.J, gamma=0.8, alpha=0.8)
        nmpc = QuadcopterNMPC(m=0.18, J=env.quad.J, dt=env.dt, N=25, num_obs=len(active_obstacles))
    elif controller_type == "NMPC-aCBF": 
        cbf = AdaptiveCBFQPController(m_nom=0.18, J=env.quad.J, gamma=0.6, alpha=0.6)
        nmpc = QuadcopterNMPC(m=0.18, J=env.quad.J, dt=env.dt, N=25, num_obs=len(active_obstacles))
    elif controller_type == "A* + LQR":
        planner_env = SimulationEnviroment(goal=goal, res=1)
        for obs in active_obstacles:
            planner_env.CreateObstacle(obs.c[0], obs.c[1], obs.r)
        
        planner_env.pathFinder.CreateGrid(planner_env.obstacle)
        raw_path = planner_env.pathFinder.PathFind(start, goal)
        
        if raw_path is None:
            raise ValueError("A* could not find a valid path!")
            
        pulled_path = planner_env.sampledPath(planner_env.PullPath(raw_path))
        traj = MinimumSnapTrajectory(pulled_path, avgSpeed=10)
        
        A = np.zeros([6,6])
        A[0,3]=1; A[1,4]=1; A[2,5]=1; A[3,2]=-9.81
        m_nom, J_nom = 0.18, env.quad.J
        B = np.zeros([6,2])
        B[4,0]=1/m_nom; B[5,1]=1/J_nom
        
        sys_d = control.c2d(control.ss(A, B, np.eye(6), np.zeros((6,2))), Ts=env.dt)
        Q_lqr = np.diag([1, 10, 50, 1, 1, 10])
        R_lqr = np.diag([0.1, 1])
        K, _, _ = control.dlqr(sys_d.A, sys_d.B, Q_lqr, R_lqr)
        
        lqr = LQRController(uEq=[m_nom*9.81, 0], m=m_nom, K=K)

    obs, _ = env.reset()
    env.quad.state = X(np.array([start[0], start[1], 0, 0, 0, 0]))
    env.goal = np.array(goal)
    env.prev_dist = np.linalg.norm(env.goal - np.array(start))
    obs = env._get_obs()
    
    history = {'x': [], 'y': [], 'phi': [], 't': [], 'est_mass': [], 'est_wind_x': [], 'est_wind_y': [], 'obs_snapshots': []}
    done, truncated, steps = False, False, 0
    crashed = False 
    
    start_calc_time = time.time()
    
    while not (done or truncated):
        t = steps * env.dt
        
        current_true_mass = scenario_dict['mass_func'](t)
        current_wind = scenario_dict['wind_func'](t)
        env.quad.m = current_true_mass
        state = env.quad.state
        
        history['x'].append(state.x)
        history['y'].append(state.y)
        history['phi'].append(state.phi)
        history['t'].append(t)
        history['obs_snapshots'].append([copy.deepcopy(o) for o in active_obstacles])
        
        gym_action = np.zeros(2)
        u1_applied = 0.0

        if controller_type in ["RL Nominal", "RL-CBF", "RL-aCBF","RL-DynCBF"]:
            action_sac, _ = model_rl.predict(obs, deterministic=True)
            est_mass = (1/ cbf.theta_m) if controller_type == "RL-aCBF" else 0.18
            F_nom = ((action_sac[0] + 1) / 2) * (1.8 * est_mass * 9.81)
            F_nom = np.clip(F_nom, 0, 1.8 * 0.18 * 9.81)
            M_nom = action_sac[1] * 0.02
            u_nom = np.array([F_nom, M_nom], dtype=np.float64)
            
            u_safe = u_nom if controller_type == "RL Nominal" else cbf.filter_action(u_nom, state, active_obstacles)
            gym_action[0] = (u_safe[0] / (1.8 * current_true_mass * 9.81)) * 2 - 1 
            gym_action[1] = u_safe[1] / 0.02
            u1_applied = ((gym_action[0] + 1) / 2) * (1.8 * current_true_mass * 9.81)

        elif controller_type == "NMPC":
            target = np.array([goal[0], goal[1]])
            u_opt = nmpc.compute_action(state, target, active_obstacles)
            gym_action[0] = (float(u_opt[0]) / (1.8 * current_true_mass * 9.81)) * 2 - 1
            gym_action[1] = float(u_opt[1]) / 0.02
            u1_applied = float(u_opt[0])

        elif controller_type == "NMPC_A*":
            future_t = min(t + 1.0, traj.totalTime)
            target_state = traj(future_t) 
            target_pos = np.array([target_state[0], target_state[1]])
            
            u_opt = nmpc.compute_action(state, target_pos, active_obstacles)
            
        elif controller_type in ["NMPC-CBF", "NMPC-aCBF"]:
            target = np.array([goal[0], goal[1]])
            u_nom = nmpc.compute_action(state, target, active_obstacles)
            u_safe = cbf.filter_action(u_nom, state, active_obstacles)
            gym_action[0] = (u_safe[0] / (1.8 * current_true_mass * 9.81)) * 2 - 1 
            gym_action[1] = u_safe[1] / 0.02
            u1_applied = u_safe[0]
            
        elif controller_type == "A* + LQR":
            target_list = traj(t)
            target_state = X(target_list)
            
            u_raw = lqr.Controller(target_state, state, saturate=False)
            
            gym_action[0] = (u_raw[0] / (1.8 * current_true_mass * 9.81)) * 2 - 1 
            gym_action[1] = u_raw[1] / 0.02
            u1_applied = u_raw[0]

        gym_action = np.clip(gym_action, -1.0, 1.0)
        
        vx_old = env.quad.state.xDot
        vy_old = env.quad.state.yDot
        phi_old = env.quad.state.phi
        
        obs, reward, done, truncated, info = env.step(gym_action, wind_force=current_wind)
        
        for obs_obj in active_obstacles:
            if hasattr(obs_obj, 'update'): obs_obj.update(env.dt)
                
        if controller_type in ["RL-aCBF", "NMPC-aCBF"]:
            true_ax = (env.quad.state.xDot - vx_old) / env.dt
            true_ay = (env.quad.state.yDot - vy_old) / env.dt
            cbf.adapt(phi_old, u1_applied, true_ax, true_ay, env.dt, vx_old, vy_old)
            history['est_mass'].append(1/ cbf.theta_m)
            history['est_wind_x'].append(cbf.d_x)
            history['est_wind_y'].append(getattr(cbf, 'd_y', 0.0))
        else:
            history['est_mass'].append(0.18)
            history['est_wind_x'].append(0.0)
            history['est_wind_y'].append(0.0)
            
        current_pos = np.array([state.x, state.y])
        for obs_obj in active_obstacles:
            if np.linalg.norm(current_pos - obs_obj.c) <= obs_obj.r:
                crashed = True
                done = True
                break
        steps += 1
        
    end_calc_time = time.time()
    
    is_success = info.get('is_success', False) if 'info' in locals() else False
    if crashed: is_success = False
        
    history['success'] = is_success
    history['flight_time'] = steps * env.dt
    history['calc_time'] = end_calc_time - start_calc_time
        
    return history, scenario_dict

def plot_static_comparison(nom_hist, adapt_hist, scenario_dict, title_text):
    start, goal = scenario_dict['start'], scenario_dict['goal']
    time_steps = np.array(adapt_hist['t'])
    
    true_masses = np.array([scenario_dict['mass_func'](t) for t in time_steps])
    true_wind_x = np.array([scenario_dict['wind_func'](t)[0] / m for t, m in zip(time_steps, true_masses)])

    fig = plt.figure(figsize=(12, 8))

    ax1 = plt.subplot(2, 1, 1)
    ax1.plot(nom_hist['x'], nom_hist['y'], 'r--', linewidth=2, label="Nominal CBF")
    ax1.plot(adapt_hist['x'], adapt_hist['y'], 'b-', linewidth=2.5, label="Adaptive CBF")
    ax1.scatter([start[0]], [start[1]], c='k', label="Start", zorder=5)
    ax1.scatter([goal[0]], [goal[1]], c='g', s=100, label="Goal", zorder=5)
    
    goal_circle = plt.Circle((goal[0], goal[1]), 25, color='g', fill=False, linestyle='--')
    ax1.add_patch(goal_circle)

    for obs_obj in scenario_dict['obstacles']:
        obs_circle = plt.Circle(obs_obj.c, obs_obj.r, color='red', alpha=0.5)
        safe_circle = plt.Circle(obs_obj.c, obs_obj.r + 5, color='red', fill=False, linestyle=':')
        ax1.add_patch(obs_circle)
        ax1.add_patch(safe_circle)

    ax1.set_xlim(0, 800)
    ax1.set_ylim(0, 450)
    ax1.set_title(title_text, fontweight='bold')
    ax1.legend()
    ax1.grid(True, linestyle=':')

    ax2 = plt.subplot(2, 2, 3)
    ax2.plot(time_steps, adapt_hist['est_mass'], 'b-', linewidth=2)
    ax2.plot(time_steps, true_masses, 'g--', label="True Mass")
    ax2.axhline(y=0.18, color='r', linestyle=':', label="Nominal Assumption (0.18kg)")
    ax2.set_title("Real-Time Mass Estimation ($\hat{m}$)")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Mass (kg)")
    ax2.legend()
    ax2.grid(True, linestyle=':')

    ax3 = plt.subplot(2, 2, 4)
    ax3.plot(time_steps, adapt_hist['est_wind_x'], 'b-', linewidth=2)
    ax3.plot(time_steps, true_wind_x, 'g--', label="True Wind Accel X")
    ax3.axhline(y=0.0, color='r', linestyle=':', label="Nominal Assumption (0.0)")
    ax3.set_title("Real-Time Wind X Estimation ($\hat{d}_x$)")
    ax3.set_xlabel("Time (s)")
    ax3.set_ylabel("Acceleration ($m/s^2$)")
    ax3.legend()
    ax3.grid(True, linestyle=':')

    plt.tight_layout()
    plt.show()

 


In [ ]:
def compare_controllers(controller_list, scenario_name, model_rl):
    print(f"\n--- Running Scenario: {scenario_name} ---")
    scenario_dict = get_scenario(scenario_name)
    results = {}
    summary_data = []
    
    for ctrl in controller_list:
        print(f"Simulating {ctrl}")
        hist, _ = run_simulation(ctrl, scenario_name, model_rl)
        results[ctrl] = hist
        
        status = "SUCCESS" if hist['success'] else "FAILED/CRASHED"
        summary_data.append([ctrl, status, f"{hist['flight_time']:.2f}", f"{hist['calc_time']:.4f}"])
        
    print("\n" + "=" * 72)
    print(f"| {'Controller':<20} | {'Status':<15} | {'Flight Time (s)':<15} | {'Calc Time (s)':<12} |")
    print("-" * 72)
    for row in summary_data:
        print(f"| {row[0]:<20} | {row[1]:<15} | {row[2]:<15} | {row[3]:<12} |")
    print("=" * 72 + "\n")
        
    return results, scenario_dict


def generate_individual_plots(results, scenario_dict, title_text):
    start, goal = scenario_dict['start'], scenario_dict['goal']
    
    style_map = {
        'A* + LQR': {'color': 'black', 'linestyle': '-.'},
        'RL Nominal':   {'color': 'red', 'linestyle': '--'},
        'RL-CBF':       {'color': 'magenta', 'linestyle': '-'},
        'RL-aCBF':      {'color': 'blue', 'linestyle': '-'},
        'RL-DynCBF':    {'color': 'purple', 'linestyle': '--'},
        'NMPC':      {'color': 'green', 'linestyle': '-'},
        'NMPC-CBF':  {'color': 'cyan', 'linestyle': '-'},
        'NMPC-aCBF': {'color': 'darkorange', 'linestyle': '-'}
    }
    
    safe_title = title_text.replace(" ", "_").replace(":", "").replace(",", "")
    
  
    plt.figure(figsize=(12, 6))
    for ctrl, hist in results.items():
        st = style_map.get(ctrl, {'color': 'gray', 'linestyle': '-'})
        plt.plot(hist['x'], hist['y'], color=st['color'], linestyle=st['linestyle'], 
                 linewidth=2.5, label=f"{ctrl} Flight Path")
        
    plt.scatter([start[0]], [start[1]], c='k', label="Start", zorder=5)
    plt.scatter([goal[0]], [goal[1]], c='g', s=100, label="Goal", zorder=5)
    plt.gca().add_patch(plt.Circle((goal[0], goal[1]), 25, color='g', fill=False, linestyle='--'))

    for obs_obj in scenario_dict['obstacles']:
        plt.gca().add_patch(plt.Circle(obs_obj.c, obs_obj.r, color='red', alpha=0.5))
        plt.gca().add_patch(plt.Circle(obs_obj.c, obs_obj.r + 5, color='red', fill=False, linestyle=':'))

    plt.xlim(0, 800)
    plt.ylim(0, 450)
    plt.title(title_text, fontweight='bold', fontsize=14)
    plt.xlabel("X Position (m)")
    plt.ylabel("Y Position (m)")
    plt.legend()
    plt.grid(True, linestyle=':')
    plt.tight_layout()
    plt.savefig(f"{safe_title}_Trajectory.pdf", format='pdf', bbox_inches='tight')
    plt.show()

   
    if 'RL-aCBF' in results or 'NMPC-aCBF' in results:
        adapt_key = 'RL-aCBF' if 'RL-aCBF' in results else 'NMPC-aCBF'
        adapt_hist = results[adapt_key]
        
        time_steps = np.array(adapt_hist['t'])
        true_masses = np.array([scenario_dict['mass_func'](t) for t in time_steps])
        true_wind_x = np.array([scenario_dict['wind_func'](t)[0] / m for t, m in zip(time_steps, true_masses)])
        '''
        plt.figure(figsize=(8, 5))
        plt.plot(time_steps, adapt_hist['est_mass'], 'b-', linewidth=2.5)
        plt.plot(time_steps, true_masses, 'g--', linewidth=2, label="True Mass")
        plt.axhline(y=0.18, color='r', linestyle=':', linewidth=2, label="Nominal Assumption (0.18kg)")
        plt.title(f"Real-Time Mass Estimation ({adapt_key})", fontweight='bold', fontsize=14)
        plt.xlabel("Time (s)", fontsize=12)
        plt.ylabel("Mass (kg)", fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"{safe_title}_Mass.pdf", format='pdf', bbox_inches='tight')
        #plt.show()

        plt.figure(figsize=(8, 5))
        plt.plot(time_steps, adapt_hist['est_wind_x'], 'b-', linewidth=2.5)
        plt.plot(time_steps, true_wind_x, 'g--', linewidth=2, label="True Wind Accel X")
        plt.axhline(y=0.0, color='r', linestyle=':', linewidth=2, label="Nominal Assumption (0$m/s^2$)")
        plt.title(f"Real-Time Wind X Estimation ({adapt_key})", fontweight='bold', fontsize=14)
        plt.xlabel("Time (s)", fontsize=12)
        plt.ylabel("Acceleration ($m/s^2$)", fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f"{safe_title}_Wind.pdf", format='pdf', bbox_inches='tight')
        #plt.show()
        '''

def generate_video(hist, scenario_dict, title="Dynamic Flight Simulation"):
    matplotlib.rcParams['animation.embed_limit'] = 500
    start, goal = scenario_dict['start'], scenario_dict['goal']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_xlim(0, 850); ax.set_ylim(0, 450)
    ax.set_title(title, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=1)
    
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
    ax.scatter([start[0]], [start[1]], c='k', label="Start", zorder=5)
    ax.scatter([goal[0]], [goal[1]], c='g', s=100, label="Goal", zorder=5)
    ax.add_patch(plt.Circle((goal[0], goal[1]), 25, color='g', fill=False, linestyle='--'))

    drone_path_line, = ax.plot([], [], 'b-', linewidth=1.5, alpha=0.5, label="Flight Path")
    drone_body, = ax.plot([], [], 'k-', linewidth=2, solid_capstyle='round') 
    drone_thrust, = ax.plot([], [], 'r-', linewidth=2) 
    wind_arrow, = ax.plot([], [], 'c-', linewidth=2.5, label="Est. Wind ($d_x, d_y$)")
    wind_head, = ax.plot([], [], 'c>', markersize=8)

    obs_patches, safe_patches = [], []
    for _ in range(len(scenario_dict['obstacles'])):
        o_patch = plt.Circle((0, 0), 0, color='r', alpha=0.5)
        s_patch = plt.Circle((0, 0), 0, color='r', fill=False, linestyle=':')
        ax.add_patch(o_patch); ax.add_patch(s_patch)
        obs_patches.append(o_patch); safe_patches.append(s_patch)

    ax.legend(loc="upper right", fontsize=10)

    def init():
        drone_path_line.set_data([], []); drone_body.set_data([], [])
        drone_thrust.set_data([], []); wind_arrow.set_data([], []); wind_head.set_data([], [])
        return [drone_path_line, drone_body, drone_thrust, wind_arrow, wind_head, time_text] + obs_patches + safe_patches

    def animate(i):
        state_x, state_y, state_phi = hist['x'][i], hist['y'][i], hist['phi'][i]
        obs_snapshot = hist['obs_snapshots'][i]
        drone_path_line.set_data(hist['x'][:i+1], hist['y'][:i+1])
        
        L = 10
        dx, dy = L/2 * np.cos(state_phi), L/2 * np.sin(state_phi)
        drone_body.set_data([state_x - dx, state_x + dx], [state_y - dy, state_y + dy])
        
        tx, ty = 8 * np.sin(state_phi), 8 * np.cos(state_phi)
        drone_thrust.set_data([state_x, state_x - tx], [state_y, state_y + ty])
                              
        wind_scale = 60
        wx, wy = hist['est_wind_x'][i] * wind_scale, hist['est_wind_y'][i] * wind_scale
        if np.linalg.norm([wx, wy]) > 1.0: 
            wind_arrow.set_data([state_x, state_x + wx], [state_y, state_y + wy])
            angle = np.degrees(np.arctan2(wy, wx))
            wind_head.set_marker((3, 0, angle - 90)) 
            wind_head.set_data([state_x + wx], [state_y + wy])
        else:
            wind_arrow.set_data([], []); wind_head.set_data([], [])
            
        for j, obs_obj in enumerate(obs_snapshot):
            obs_patches[j].center = (obs_obj.c[0], obs_obj.c[1])
            obs_patches[j].set_radius(obs_obj.r)
            safe_patches[j].center = (obs_obj.c[0], obs_obj.c[1])
            safe_patches[j].set_radius(obs_obj.r + 10)
            
        time_text.set_text(f"Time: {hist['t'][i]:.2f} s")
        return [drone_path_line, drone_body, drone_thrust, wind_arrow, wind_head, time_text] + obs_patches + safe_patches

    ani = animation.FuncAnimation(fig, animate, init_func=init, frames=range(0, len(hist['x']), 2), interval=40, blit=True)
    plt.close(fig) 
    return HTML(ani.to_jshtml())


def generate_combined_video(results, scenario_dict, title="Combined Dynamic Flight Simulation"):
    matplotlib.rcParams['animation.embed_limit'] = 500
    start, goal = scenario_dict['start'], scenario_dict['goal']
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_xlim(0, 850); ax.set_ylim(0, 450)
    ax.set_title(title, fontweight='bold', fontsize=14)
    ax.grid(True, linestyle=':', alpha=1)
    
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
    ax.scatter([start[0]], [start[1]], c='k', label="Start", zorder=5)
    ax.scatter([goal[0]], [goal[1]], c='g', s=100, label="Goal", zorder=5)
    ax.add_patch(plt.Circle((goal[0], goal[1]), 25, color='g', fill=False, linestyle='--'))

    style_map = {
        'A* + LQR': {'color': 'black', 'linestyle': '-.'},
        'RL Nominal':   {'color': 'red', 'linestyle': '--'},
        'RL-CBF':       {'color': 'magenta', 'linestyle': '-'},
        'RL-aCBF':      {'color': 'blue', 'linestyle': '-'},
        'RL-DynCBF':    {'color': 'purple', 'linestyle': '--'},
        'NMPC':      {'color': 'green', 'linestyle': '-'},
        'NMPC-CBF':  {'color': 'cyan', 'linestyle': '-'},
        'NMPC-aCBF': {'color': 'darkorange', 'linestyle': '-'}
    }
    
    drone_graphics = {}
    for ctrl in results.keys():
        st = style_map.get(ctrl, {'color': 'gray', 'linestyle': '-'})
        
        path, = ax.plot([], [], color=st['color'], linestyle=st['linestyle'], linewidth=2.0, alpha=0.6, label=ctrl)
        
        body, = ax.plot([], [], color=st['color'], linestyle='-', linewidth=2.5, solid_capstyle='round')
        
        thrust, = ax.plot([], [], color='black', linestyle='-', linewidth=1.5)
        
        wind_arrow, wind_head = None, None
        if 'aCBF' in ctrl:
            wind_arrow, = ax.plot([], [], color=st['color'], linestyle='-', linewidth=3, alpha=0.4, label="_nolegend_")
            wind_head, = ax.plot([], [], color=st['color'], marker='>', markersize=5, alpha=0.4, label="_nolegend_")

        drone_graphics[ctrl] = {
            'path': path, 
            'body': body, 
            'thrust': thrust,
            'wind_arrow': wind_arrow, 
            'wind_head': wind_head
        }

    obs_patches, safe_patches = [], []
    for _ in range(len(scenario_dict['obstacles'])):
        o_patch = plt.Circle((0, 0), 0, color='red', alpha=0.5)
        s_patch = plt.Circle((0, 0), 0, color='red', fill=False, linestyle=':')
        ax.add_patch(o_patch); ax.add_patch(s_patch)
        obs_patches.append(o_patch); safe_patches.append(s_patch)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), loc="upper right", fontsize=10)

    longest_ctrl = max(results.keys(), key=lambda k: len(results[k]['x']))
    max_frames = len(results[longest_ctrl]['x'])
    ref_hist = results[longest_ctrl]

    def init():
        ret = []
        for g in drone_graphics.values():
            g['path'].set_data([], [])
            g['body'].set_data([], [])
            g['thrust'].set_data([], [])
            ret.extend([g['path'], g['body'], g['thrust']])
            
            if g['wind_arrow'] is not None:
                g['wind_arrow'].set_data([], [])
                g['wind_head'].set_data([], [])
                ret.extend([g['wind_arrow'], g['wind_head']])
                
        return ret + [time_text] + obs_patches + safe_patches

    def animate(i):
        for ctrl, hist in results.items():
            idx = min(i, len(hist['x']) - 1) 
            state_x, state_y, state_phi = hist['x'][idx], hist['y'][idx], hist['phi'][idx]
            
            drone_graphics[ctrl]['path'].set_data(hist['x'][:idx+1], hist['y'][:idx+1])
            
            L = 10
            dx, dy = L/2 * np.cos(state_phi), L/2 * np.sin(state_phi)
            drone_graphics[ctrl]['body'].set_data([state_x - dx, state_x + dx], [state_y - dy, state_y + dy])
            
            tx, ty = 8 * np.sin(state_phi), 8 * np.cos(state_phi)
            drone_graphics[ctrl]['thrust'].set_data([state_x, state_x - tx], [state_y, state_y + ty])
            
            if drone_graphics[ctrl]['wind_arrow'] is not None:
                wind_scale = 80 
                wx, wy = hist['est_wind_x'][idx] * wind_scale, hist['est_wind_y'][idx] * wind_scale
                
                if np.linalg.norm([wx, wy]) > 1.0: 
                    drone_graphics[ctrl]['wind_arrow'].set_data([state_x, state_x + wx], [state_y, state_y + wy])
                    angle = np.degrees(np.arctan2(wy, wx))
                    drone_graphics[ctrl]['wind_head'].set_marker((3, 0, angle - 90)) 
                    drone_graphics[ctrl]['wind_head'].set_data([state_x + wx], [state_y + wy])
                else:
                    drone_graphics[ctrl]['wind_arrow'].set_data([], [])
                    drone_graphics[ctrl]['wind_head'].set_data([], [])

        ref_idx = min(i, len(ref_hist['obs_snapshots']) - 1)
        obs_snapshot = ref_hist['obs_snapshots'][ref_idx]
        
        for j, obs_obj in enumerate(obs_snapshot):
            obs_patches[j].center = (obs_obj.c[0], obs_obj.c[1])
            obs_patches[j].set_radius(obs_obj.r)
            safe_patches[j].center = (obs_obj.c[0], obs_obj.c[1])
            safe_patches[j].set_radius(obs_obj.r + 10)
            
        time_text.set_text(f"Time: {ref_hist['t'][ref_idx]:.2f} s")
        
        ret = []
        for g in drone_graphics.values():
            ret.extend([g['path'], g['body'], g['thrust']])
            if g['wind_arrow'] is not None:
                ret.extend([g['wind_arrow'], g['wind_head']])
                
        return ret + [time_text] + obs_patches + safe_patches

    ani = animation.FuncAnimation(fig, animate, init_func=init, frames=range(0, max_frames, 2), interval=40, blit=True)
    plt.rcParams['animation.ffmpeg_path'] = r"C:\FFmpeg\bin\ffmpeg.exe"
    Writer = animation.writers['ffmpeg']
    writer = Writer(fps=25, metadata=dict(artist='Nia Touko'), bitrate=1800)
    ani.save(title+".mp4", writer=writer)
    plt.close(fig) 
    return HTML(ani.to_jshtml())

In [ ]:
from IPython.display import display

all_controllers = [
    "A* + LQR", "RL Nominal", "RL-CBF", "RL-aCBF", "RL-DynCBF",
    "NMPC", "NMPC-aCBF"
]

all_controllers_adapt =[
    "A* + LQR", "RL Nominal", "RL-CBF", "RL-aCBF",
    "NMPC", "NMPC-aCBF"
]


print("\n" + "="*50)
print("TEST CASE 1: STATIC OBSTACLES")
results_static_single, scenario_static_single = compare_controllers(
    all_controllers, 
    "Static_Single", 
    model
)
generate_individual_plots(
    results_static_single, 
    scenario_static_single, 
    "Static Single Obstacle Avoidance (All Models)"
)

print("\n" + "="*50)
results_static_multi, scenario_static_multi = compare_controllers(
    all_controllers, 
    "Static_Multi", 
    model
)
generate_individual_plots(
    results_static_multi, 
    scenario_static_multi, 
    "Static Multi-Obstacle Avoidance (All Models)"
)

print("\n" + "="*50)
print("TEST CASE 2: DYNAMIC OBSTACLES")
print("="*50)
results_dynamic, scenario_dynamic = compare_controllers(
    all_controllers, 
    "Dynamic_Single", 
    model
)
generate_individual_plots(
    results_dynamic, 
    scenario_dynamic, 
    "Dynamic Single Obstacle Avoidance Comparison (All Models)"
)
'''
print("Generating combined video for dynamic environment")
video_html_dyn = generate_combined_video(
    results_dynamic, 
    scenario_dynamic, 
    "Dynamic Single Obstacle Race (All Models)"
)
'''
results_dynamic, scenario_dynamic = compare_controllers(
    all_controllers, 
    "Dynamic_Hard", 
    model
)
generate_individual_plots(
    results_dynamic, 
    scenario_dynamic, 
    "Dynamic Obstacle Avoidance Comparison (All Models)"
)
'''
print("Generating combined video for dynamic environment")
video_html_dyn = generate_combined_video(
    results_dynamic, 
    scenario_dynamic, 
    "Dynamic Multi-Obstacle Race (All Models)"
)
display(video_html_dyn)
'''
print("\n" + "="*50)
print("TEST CASE 3: SEVERE PARAMETER MISMATCH")
print("="*50)
results_loss, scenario_loss = compare_controllers(
    all_controllers, 
    "Loss_of_Parameter_1", 
    model
)
generate_individual_plots(
    results_loss, 
    scenario_loss, 
    "Severe Parameter Mismatch Comparison (Mass: 0.28kg, Wind: 0.15N)"
)
'''
print("Generating combined video for parameter mismatch")
video_html_loss = generate_combined_video(
    results_loss, 
    scenario_loss, 
    "Parameter Mismatch Survival (All Models)"
)
display(video_html_loss)
'''

print("\n" + "="*50)
print("TEST CASE 4: MID-FLIGHT DYNAMIC SHIFT")
print("="*50)
results_shift, scenario_shift = compare_controllers(
    all_controllers, 
    "Dynamic_Shift", 
    model
)
generate_individual_plots(
    results_shift, 
    scenario_shift, 
    "Mid-Flight Parameter Shift Comparison (All Models)"
)
'''
print("Generating combined video for mid-flight parameter shift")
video_html_shift = generate_combined_video(
    results_shift, 
    scenario_shift, 
    "Mid-Flight Shift Adaptation (All Models)"
)

display(video_html_shift)

'''

In [ ]:

print("\n" + "="*50)
print("TEST CASE 5: DYNAMIC HARD + UNCERTAINTY")
print("="*50)
results_dyn_unc, scenario_dyn_unc = compare_controllers(
    all_controllers, 
    "Dynamic_Hard_Uncertainty", 
    model
)

generate_individual_plots(
    results_dyn_unc, 
    scenario_dyn_unc, 
    "Dynamic Multi-Obstacle Avoidance under Severe Parameter Shift"
)


In [ ]:
'''
print("Generating combined video for dynamic uncertainty")
video_html_dyn_unc = generate_combined_video(
    results_dyn_unc, 
    scenario_dyn_unc, 
    "Dynamic Multi-Obstacle Race under Severe Parameter Shift"
)
display(video_html_dyn_unc)
'''

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

class DynamicObstacle:
    def __init__(self, x, y, r, vx=0.0, vy=0.0):
        self.x, self.y, self.r = x, y, r
        self.vx, self.vy = vx, vy

scenarios = {
    "dynamic_obstacle_case_1": [DynamicObstacle(400, 225, 80, vx=0.0, vy=15.0)],
    "dynamic_obstacle_case_2": [
        DynamicObstacle(300, 100, 50, vx=0.0, vy=15.0),
        DynamicObstacle(500, 350, 50, vx=0.0, vy=-15.0),
        DynamicObstacle(400, 225, 40, vx=10.6, vy=10.6)
    ],
    "dynamic_obstacle_case_3": [
        DynamicObstacle(350, 70,  60, vx=0.0, vy=15.0),
        DynamicObstacle(650, 225, 40, vx=0.0, vy=-15.0),
        DynamicObstacle(400, 175, 50, vx=10.6, vy=10.6),
        DynamicObstacle(200, 350, 40, vx=15.0, vy=0.0),
        DynamicObstacle(200, 50,  40, vx=6.7, vy=13.4)
    ]
}

start = (50, 225)
goal = (750, 200)

for name, obstacles in scenarios.items():
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=300)
    ax.set_xlim(0, 800)
    ax.set_ylim(0, 450)
    
    ax.plot(*start, 'ko', markersize=8, label='Start Node', zorder=5)
    ax.plot(*goal, 'go', markersize=10, label='Goal Node', zorder=5)
    
    goal_circle = patches.Circle(goal, 25, color='green', fill=False, linestyle='--', zorder=4)
    ax.add_patch(goal_circle)

    for i, obs in enumerate(obstacles):
        obs_circle = patches.Circle((obs.x, obs.y), obs.r, color='red', alpha=0.5, zorder=3)
        ax.add_patch(obs_circle)
        
        safe_circle = patches.Circle((obs.x, obs.y), obs.r + 5, color='red', fill=False, linestyle=':', zorder=3)
        ax.add_patch(safe_circle)
        
        scale = 3
        if obs.vx != 0 or obs.vy != 0:
            ax.arrow(obs.x, obs.y, obs.vx * scale, obs.vy * scale, 
                     head_width=15, head_length=15, fc='blue', ec='blue', 
                     linewidth=2, zorder=5)
            
            v_mag = np.sqrt(obs.vx**2 + obs.vy**2)
            
            text_x = obs.x + (obs.vx * scale) + 10
            text_y = obs.y + (obs.vy * scale) + 10
            
            ax.text(text_x, text_y, f"{v_mag:.1f} m/s", color='blue', 
                    fontsize=10, fontweight='bold', zorder=6,
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

    ax.arrow(0, 0, 0, 0, head_width=0, head_length=0, fc='blue', ec='blue', label='Velocity Vector')

    plt.title(f'{name.replace("_", " ").title()}', fontsize=14, fontweight='bold')
    plt.xlabel('Global X Position (m)', fontsize=12)
    plt.ylabel('Global Y Position (m)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(loc='upper right', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(f'{name}.pdf', format='pdf', bbox_inches='tight')
    plt.close()

print("PDF generation complete! Check your directory.")




In [ ]:
print("\n--- COMPUTATIONAL TRACTABILITY REPORT ---")
print(f"{'Controller':<15} | {'Total Time (s)':<15} | {'Steps':<10} | {'ms / step':<10} | {'Max Hz':<10}")
print("-" * 65)

for ctrl, hist in results_dynamic.items(): 
    total_calc_time = hist['calc_time']
    steps = len(hist['x'])
    ms_per_step = (total_calc_time / steps) * 1000
    max_hz = 1000 / ms_per_step if ms_per_step > 0 else 0
    
    print(f"{ctrl:<15} | {total_calc_time:<15.4f} | {steps:<10} | {ms_per_step:<10.2f} | {max_hz:<10.0f}")



